# NABirds classifier model

This notebook trains and EfficientNet-B4 classifier using transfer learning on the NABirds dataset. First, the classifier head is trained with the model backbone frozen. Then, the entire network is trained using a lower learning rate. The DeepLake NABirds training data is split into training, validation (15%), and test (15%) data for training and experiment evaluation, and the DeepLake NABirds validation data is reserved as holdout data for final testing. As in the fewshot experiments in the previous notebook, accuracy is used as the key metric while precision, recall, and f1 score are also tracked.

Notebook structure:
- Setup and configuration
- Initial processing visualization
- Training preparation
- Tuning classifier head
- Fine tuning the classifier backbone
- Hyperparameter tuning
- Final training
- Holdout dataset evaluation


# Setup and configuration:

A GPU from Google Colab is used for this project for model training, so relevant packages are installed in the environment, and relevant files (model_tune.py, duplicate removing index files) are linked from Github:

In [ ]:
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Torrtheb/data-science-portfolio.git"
BRANCH = "bird-classifier-clean"
RAW_BASE_URL = f"https://raw.githubusercontent.com/Torrtheb/data-science-portfolio/{BRANCH}/bird_classifier"

CLEANED_INDEX_FILES = [
    "label_index_train_clean.npz",
    "label_index_val_clean.npz",
    "clean_indices.npz",
]

PYTHON_MODULES = ["eda", "fewshot", "model_tune", "utilities_fewshot"]

if IN_COLAB:
    %pip install -q imagehash "deeplake<4" umap-learn albumentations \
    opencv-python-headless scikit-image lime scikit-learn seaborn \
    jupyter-black optuna

    REPO_PATH = "/content/repo"
    PROJECT_PATH = f"{REPO_PATH}/bird_classifier"
    MODULE_PATH = f"{PROJECT_PATH}/python_files"

    !rm -rf {REPO_PATH}
    !git clone --branch {BRANCH} --single-branch --quiet {REPO_URL} {REPO_PATH}

    if MODULE_PATH not in sys.path:
        sys.path.insert(0, MODULE_PATH)

    for mod in PYTHON_MODULES:
        if mod in sys.modules:
            del sys.modules[mod]

    print("\n Downloading cleaned index files from GitHub...")
    for filename in CLEANED_INDEX_FILES:
        dest_path = Path(PROJECT_PATH) / filename
        url = f"{RAW_BASE_URL}/{filename}"
        try:
            urllib.request.urlretrieve(url, str(dest_path))
            print(f"   {filename}")
        except Exception as e:
            print(f"   {filename} not available (run eda.ipynb locally first): {e}")

    print("\n Colab setup complete")
else:
    MODULE_PATH = str(Path("./python_files").resolve())
    if MODULE_PATH not in sys.path:
        sys.path.insert(0, MODULE_PATH)
    print("Running locally")

Importing libraries:

In [ ]:
import random
import os
import time as _time
from typing import Dict, List, Tuple, Optional, Any
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix as sklearn_cm,
)
import jupyter_black
import json
import albumentations as A
import cv2
import deeplake
from model_tune import (
    _ensure_uint8_rgb,
    _pad_to_square,
    get_device,
    seed_everything,
    DEVICE,
    RUNS_DIR,
    build_label_index,
    save_label_index,
    load_label_index,
    SplitIndices,
    split_label_index_stratified,
    apply_bbox_crop_optimized,
    BirdDataset,
    build_efficientnet_b4,
    make_dataloader,
    evaluate,
    evaluate_with_preds,
    TrainConfig,
    train_two_stage,
    visualize_preprocessing_comparison,
    plot_training_history,
    make_experiment_runner,
    get_completed_experiments,
    resolve_train_augmentation,
    create_tuning_subset,
    aug_objective,
    create_fast_objective,
    get_class_name,
    compute_confused_pairs,
    prepare_lime_image,
    compute_confused_pairs
)

%matplotlib inline
try:
    import optuna
    from optuna.pruners import MedianPruner
except ImportError:
    print("Installing optuna...")
    %pip install -q optuna
    import optuna
    from optuna.pruners import MedianPruner

try:
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
except ImportError:
    print("Installing LIME...")
    %pip install -q lime
    from lime import lime_image
    from skimage.segmentation import mark_boundaries
jupyter_black.load()
print(" All libraries imported successfully")
print(f" Module path: {MODULE_PATH}")

Ensuring that a GPU from Colab is available, mounting google drive for caching, setting a reasonable batch size for allowed memory, and setting up notebook reproducibility:

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    RUNS_DIR = Path("/content/drive/MyDrive/bird_classifier_runs")
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted - results will persist!")
    print(f"  Runs directory: {RUNS_DIR}")
else:
    from model_tune import RUNS_DIR

    print(f"Running locally, runs directory: {RUNS_DIR}")

if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Available: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.fp32_precision = "tf32"
    torch.backends.cudnn.conv.fp32_precision = "tf32"
    print("   CUDA optimizations enabled (cuDNN benchmark, TF32)")
    if gpu_mem_gb >= 15:
        BATCH_SIZE = 64
    elif gpu_mem_gb >= 8:
        BATCH_SIZE = 32
    else:
        BATCH_SIZE = 16
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    BATCH_SIZE = 32
    print("Using Apple MPS")
else:
    BATCH_SIZE = 16
    print(" Using CPU (training will be slow)")

print(f" Using device: {DEVICE} (batch size: {BATCH_SIZE})")
SEED = 42
seed_everything(SEED)
print(f"Random seed: {SEED}")
DATA_ROOT = Path("/content/data") if IN_COLAB else Path("data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Data root: {DATA_ROOT}")

Setting up caching:

In [ ]:
USE_PERSISTENT_CACHE = True

if USE_PERSISTENT_CACHE:
    if IN_COLAB:
        CACHE_DIR = Path("/content/drive/MyDrive/bird_classifier_cache")
    else:
        CACHE_DIR = Path(
            os.environ.get("BIRD_CACHE_DIR", str(Path.home() / "bird_classifier_cache"))
        )
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Cache directory: {CACHE_DIR}")
else:
    CACHE_DIR = None
    print("Caching disabled")

## Loading data:

The data is loaded from deeplake, and a class label index is built with any duplicates removed, as seen in eda.ipynb:

In [ ]:
print("Loading datasets from DeepLake...")
ds_train = deeplake.load("hub://activeloop/nabirds-dataset-train", read_only=True)
print(f"Training samples: {len(ds_train)}")

ds_val = deeplake.load("hub://activeloop/nabirds-dataset-val", read_only=True)
print(f"Holdout samples: {len(ds_val)}")

TRAIN_CLEAN_INDEX_CANDIDATES = [
    Path(MODULE_PATH) / "label_index_train_clean.npz",
    Path("label_index_train_clean.npz"),
]
VAL_CLEAN_INDEX_CANDIDATES = [
    Path(MODULE_PATH) / "label_index_val_clean.npz",
    Path("label_index_val_clean.npz"),
]
FALLBACK_TRAIN_INDEX_CANDIDATES = [
    Path(MODULE_PATH) / "label_index_train.npz",
    Path("label_index_train.npz"),
]

TRAIN_CLEAN_INDEX_PATH = next(
    (p for p in TRAIN_CLEAN_INDEX_CANDIDATES if p.exists()), None
)
VAL_CLEAN_INDEX_PATH = next((p for p in VAL_CLEAN_INDEX_CANDIDATES if p.exists()), None)

if TRAIN_CLEAN_INDEX_PATH is not None:
    train_label_index = load_label_index(TRAIN_CLEAN_INDEX_PATH)
    total_samples = sum(len(v) for v in train_label_index.values())
    print(f"Loaded cleaned training label index from {TRAIN_CLEAN_INDEX_PATH}")
    print(f"   {total_samples:,} samples (duplicates removed)")
else:
    LABEL_INDEX_PATH = next(
        (p for p in FALLBACK_TRAIN_INDEX_CANDIDATES if p.exists()), None
    )
    if LABEL_INDEX_PATH is not None:
        train_label_index = load_label_index(LABEL_INDEX_PATH)
        print(f"Cleaned index not found, loaded original from {LABEL_INDEX_PATH}")
        print(
            f"   Run eda.ipynb first to generate cleaned indices (without duplicates)"
        )
    else:
        print("Building training label index (first time only)...")
        train_label_index = build_label_index(ds_train)
        save_path = Path("label_index_train.npz")
        save_label_index(train_label_index, save_path)
        print(f"Saved training label index to {save_path}")
        print("   Run eda.ipynb to generate cleaned indices (without duplicates)")

BUILD_HOLDOUT_INDEX_NOW = True
holdout_label_index = None

if BUILD_HOLDOUT_INDEX_NOW:
    if VAL_CLEAN_INDEX_PATH is not None:
        holdout_label_index = load_label_index(VAL_CLEAN_INDEX_PATH)
        total_holdout = sum(len(v) for v in holdout_label_index.values())
        print(f"Loaded CLEANED holdout label index from {VAL_CLEAN_INDEX_PATH}")
        print(f"   {total_holdout:,} samples (duplicates removed)")
    else:
        HOLDOUT_INDEX_PATH = Path("label_index_holdout.npz")
        if HOLDOUT_INDEX_PATH.exists():
            holdout_label_index = load_label_index(HOLDOUT_INDEX_PATH)
            print(
                f"Cleaned holdout index not found, loaded original from {HOLDOUT_INDEX_PATH}"
            )
        else:
            print("Building holdout label index (first time only)...")
            holdout_label_index = build_label_index(ds_val)
            save_label_index(holdout_label_index, HOLDOUT_INDEX_PATH)
            print(f"Saved holdout label index to {HOLDOUT_INDEX_PATH}")
else:
    print("Skipping holdout label index creation for now.")

CLASS_IDS = sorted(train_label_index.keys())
CLASS_ID_TO_IDX = {cid: i for i, cid in enumerate(CLASS_IDS)}
NUM_CLASSES = len(CLASS_IDS)

print(f"\nNumber of classes: {NUM_CLASSES}")

## Dataset Splitting

The same number of classes, and training samples as in the eda notebook are seen. The validation data is used as test data (no separate test dataset is present for this dataset), and will not be loaded until model evaluation. Therefore, the training data is split into separate training (70%), validation (15%), and test (15%) data. The validation data is used for checkpoint evaluation, and the test data is used for experiment evaluation.

In [ ]:
VAL_FRAC = 0.15
TEST_FRAC = 0.15

splits = split_label_index_stratified(
    train_label_index,
    class_id_to_idx=CLASS_ID_TO_IDX,
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SEED,
)

total_cleaned_samples = sum(len(v) for v in train_label_index.values())
total_raw_samples = len(ds_train)
duplicates_removed = total_raw_samples - total_cleaned_samples

train_pct = len(splits.train_idx) / total_cleaned_samples * 100
val_pct = len(splits.val_idx) / total_cleaned_samples * 100
test_pct = len(splits.test_idx) / total_cleaned_samples * 100

print("DATA SPLIT SUMMARY")
print(f"\nOriginal training data (ds_train): {total_raw_samples:,} raw samples")
if duplicates_removed > 0:
    print(f"   Duplicates removed in EDA:       {duplicates_removed:,} samples")
    print(f"   Cleaned training data:           {total_cleaned_samples:,} samples")
print(f"\n   Split from cleaned data:")
print(f"   -- Train: {len(splits.train_idx):>6,} samples ({train_pct:.1f}%)")
print(
    f"   -- Val:   {len(splits.val_idx):>6,} samples ({val_pct:.1f}%) - for checkpoint selection"
)
print(
    f"   -- Test:  {len(splits.test_idx):>6,} samples ({test_pct:.1f}%) - for intermediate evaluation"
)
print()
print(f"Holdout (ds_val): {len(ds_val):,} samples")
print()
print("Class distribution check:")
print(f"   Unique classes in train split: {len(np.unique(splits.train_y))}")
print(f"   Unique classes in val split:   {len(np.unique(splits.val_y))}")
print(f"   Unique classes in test split:  {len(np.unique(splits.test_y))}")

All splits have the same number of classes.

# Initial processing visualization

Two initial preprocessing methods are tested:
- resizing the short image edge, center crop, and ImageNet normalization
- crops to the bird's bounding box (+15% padding), then applies the backbone's native ImageNet preprocessing

In [ ]:
sample_classes = list(train_label_index.keys())[:5]
viz_indices = [
    int(train_label_index[c][0])
    for c in sample_classes
    if len(train_label_index[c]) > 0
]

print("Visualizing preprocessing modes")
print("   - Native: pads to square, then resizes to 380x380")
print("   - Bbox crop: crops to bird bbox + padding, then same as native")
print()
visualize_preprocessing_comparison(ds_train, viz_indices[:5])

As expected, the bounding box cropping ensures that the bird is centered in the image, and that minimal background is present. This sometimes reflects the bird, as the bird is often quite small relative to the whole image.

# Training preparation:

Any completed experiments are cached to Google Drive to persist results:

In [ ]:
completed = get_completed_experiments(runs_dir=RUNS_DIR)
print("=" * 60)
print("COMPLETED EXPERIMENTS (from Google Drive)")
print("=" * 60)
print(f"Runs directory: {RUNS_DIR}")
print(f"Completed experiments: {len(completed)}")
for name in completed:
    print(f"  {name}")

if not completed:
    print("No files found.")
print()
print("Note: run_or_load() will skip training for completed experiments.")

The experiment runner is initialized to coordinate training experiments:

In [ ]:
runner = make_experiment_runner(
    ds_train=ds_train,
    train_label_index=train_label_index,
    splits=splits,
    ds_holdout=ds_val,
    holdout_label_index=holdout_label_index,
    output_dir=RUNS_DIR,
)

print("Experiment runner created")
print(f"   Runs directory: {runner.runs_dir}")
print(
    f"   Splits: train={len(splits.train_idx)}, val={len(splits.val_idx)}, test={len(splits.test_idx)}"
)
print(f"   Holdout (ds_val): {len(ds_val)} samples (only for final evaluation)")

Now, the EfficientNet-B4's classifier head can be tuned using the training data.

# Tuning classifier head

Running the head-only training experiments, with original image cropping:

In [ ]:
if "runner" not in globals():
    raise RuntimeError("Run the 'EXPERIMENT RUNNER' to create runner.")

print("=" * 70)
print("EXPERIMENT 1: HEAD-ONLY TRAINING (Backbone Frozen)")
print("=" * 70)
print("Using run_or_load() - will skip training if results exist on disk")

HEAD_ONLY_EPOCHS = 10

print("\n" + "-" * 70)
print("Running: exp1a_native_head_only (1/2)")
print("-" * 70)
_t0 = _time.time()
runner.run_or_load(
    "exp1a_native_head_only",
    preprocess_mode="native",
    head_epochs=HEAD_ONLY_EPOCHS,
    finetune_epochs=0,
    cache_dir=str(CACHE_DIR) if CACHE_DIR else None,
    plot=True,
)
print(f"Completed in {(_time.time() - _t0)/60:.1f} min")

Initial validation accuracy for the first experiment is about 67%. The gap between training and validation curves shows that the model is overfitting to the training data. Examining initial results with preprocessing using the dataset’s bounding boxes:

In [ ]:
print("\n" + "-" * 70)
print("Running: exp1a_bbox_head_only (2/2)")
print("-" * 70)
_t0 = _time.time()
runner.run_or_load(
    "exp1a_bbox_head_only",
    preprocess_mode="bbox_crop",
    head_epochs=HEAD_ONLY_EPOCHS,
    finetune_epochs=0,
    cache_dir=str(CACHE_DIR) if CACHE_DIR else None,
    plot=True,
)
print(f"Completed in {(_time.time() - _t0)/60:.1f} min")

head_only_results = runner.df(sort_by="val_best_f1")
print("\n" + "=" * 70)
print("EXPERIMENT 1 RESULTS: HEAD-ONLY (ranked by val_best_f1)")
print("=" * 70)
print(
    head_only_results[
        [
            "run_name",
            "preprocess_mode",
            "val_best_f1",
            "test_f1",
            "test_acc",
            "test_top5_acc",
        ]
    ].to_string(index=False)
)

best_mode_head = head_only_results.iloc[0]["preprocess_mode"]
print(f"\n Best preprocessing (head-only): {best_mode_head}")

As for the few shot model, the bounding box processing method is best, as baseline accuracy is now almost 75% for validation data, and 72% for the test data (15% of initial training data). Generally, precision is a little higher than recall.

# Fine tuning the classifier backbone

Now, fine tuning the classifier backbone, keeping everything else the same as in the last experiment:

In [ ]:
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 15

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT 2: FULL FINE-TUNING (Head + Backbone)")
print("=" * 70)
print("Two-stage training: head first (5 epochs), then fine-tune all (15 epochs).")
print("Using run_or_load() - will skip training if results exist on disk")
print("Comparing preprocessing methods:")
print("  - native: EfficientNet transforms")
print("  - bbox_crop: Crop to bird bbox -> then EfficientNet transforms")
print()
print("\n" + "-" * 70)
print(" RUNNING: exp2a_native_finetune (1/2) - 20 total epochs")
print("-" * 70)
_native_head_ckpt = RUNS_DIR / "exp1a_native_head_only" / "best_head.pt"
_native_resume = str(_native_head_ckpt) if _native_head_ckpt.exists() else None

_t0 = _time.time()
runner.run_or_load(
    "exp2a_native_finetune",
    preprocess_mode="native",
    head_epochs=0 if _native_resume else HEAD_EPOCHS,
    resume_head_ckpt=_native_resume,
    finetune_epochs=FINETUNE_EPOCHS,
    cache_dir=str(CACHE_DIR) if CACHE_DIR else None,
    plot=True,
)
print(f"  Completed in {(_time.time() - _t0)/60:.1f} min")

For the native ImageNet processed experiment, validation accuracy has increased about 2%, to 66%, with some overfitting. Accuracy and precision are a little higher than F1 score and recall. Fine tuning the model for the bounding box preprocessing method gives:

In [ ]:
print("\n" + "-" * 70)
print(" RUNNING: exp2a_bbox_finetune (2/2) - 20 total epochs")
print("-" * 70)
_bbox_head_ckpt = RUNS_DIR / "exp1a_bbox_head_only" / "best_head.pt"
_bbox_resume = str(_bbox_head_ckpt) if _bbox_head_ckpt.exists() else None

_t0 = _time.time()
runner.run_or_load(
    "exp2a_bbox_finetune",
    preprocess_mode="bbox_crop",
    head_epochs=0 if _bbox_resume else HEAD_EPOCHS,
    resume_head_ckpt=_bbox_resume,
    finetune_epochs=FINETUNE_EPOCHS,
    cache_dir=str(CACHE_DIR) if CACHE_DIR else None,
    plot=True,
)
print(f"  Completed in {(_time.time() - _t0)/60:.1f} min")
finetune_results = runner.df(sort_by="val_best_f1")
finetune_only = finetune_results[finetune_results["run_name"].str.contains("finetune")]
print("\n" + "=" * 70)
print("EXPERIMENT 2 RESULTS: FULL FINE-TUNING (ranked by val_best_f1)")
print("=" * 70)
print(
    finetune_only[
        [
            "run_name",
            "preprocess_mode",
            "val_best_f1",
            "test_f1",
            "test_acc",
            "test_top5_acc",
        ]
    ].to_string(index=False)
)

best_mode_finetune = finetune_only.iloc[0]["preprocess_mode"]
print(f"\n Best preprocessing (fine-tuning): {best_mode_finetune}")

The fine tuned model using the bounding box approach is the best, as validation accuracy has increased to 83%, and test accuracy is 82%. This method also has the highest F1 score seen so far (79% on the test set). The best results were found on the last epoch, which shows that training could have continued. Overfitting is still seen between the training and validation data, and accuracy and precision have been consistently slightly higher than recall and F1 score. Using the bounding box preprocessing which shows that bird localization consistently helps model performance, and is used from now on.

# Hyperparameter Tuning

Now that the baseline has been established, Bayesian hyperparameter tuning with Optuna is performed to increase model metrics as much as possible. Because training on the full dataset takes a long time, a stratified subset using 20% of the initial training data is used for all hyperparameter tuning. This is split into 80% of the subset used for training, and 20% used for Optuna validation. Moreover, a small number of epochs (3-5) are used per trial to get an idea of the best model parameters without taking too long for tuning.

First, an augmentation search is performed to find the best image augmentations (image flip, rotation, brightness, blur, noise, cutout) to generalize to new data. Once optimal image processing parameters have been found, Optuna is used to find the best model parameters (for example: learning rate, weight decay, loss function, etc).

After tuning, the best hyperparameters will be used to train the final model on the full dataset for evaluation on the holdout set.

First, creating the tuning subset:

In [ ]:
DeepLakeDataset = deeplake.Dataset
TUNING_SUBSET_FRAC = 0.20

tuning_splits, tuning_label_index = create_tuning_subset(
    train_label_index,
    CLASS_ID_TO_IDX,
    subset_frac=TUNING_SUBSET_FRAC,
    val_frac=0.20,
    test_frac=0.0,
    seed=SEED,
)

total_tuning = len(tuning_splits.train_idx) + len(tuning_splits.val_idx)
full_total = len(splits.train_idx) + len(splits.val_idx) + len(splits.test_idx)
speedup = full_total / total_tuning

print("=" * 60)
print("TUNING SUBSET CREATED")
print("=" * 60)
print(f"Full dataset:    {full_total:,} samples")
print(f"Tuning subset:   {total_tuning:,} samples ({TUNING_SUBSET_FRAC*100:.0f}%)")
print(f"  -- Train:     {len(tuning_splits.train_idx):,}")
print(f"  -- Val:       {len(tuning_splits.val_idx):,}")
print(f"Classes preserved: {len(np.unique(tuning_splits.train_y))}/{NUM_CLASSES}")

All classes are preserved. Initializing the experiment:

In [ ]:
fast_runner = make_experiment_runner(
    ds_train=ds_train,
    train_label_index=tuning_label_index,
    splits=tuning_splits,
    ds_holdout=None,
    holdout_label_index=None,
    output_dir=RUNS_DIR,
)

print("Fast experiment runner created (using tuning subset)")
print(f"  Runs directory: {fast_runner.runs_dir}")
print(f"  Train: {len(tuning_splits.train_idx):,} samples")
print(f"  Val:   {len(tuning_splits.val_idx):,} samples")

## Augmentation preview

Before tuning image processing augmentations, each category in the search space is visualized to ensure that the preprocessing options will not hurt the data quality:

In [ ]:
np.random.seed(SEED)
demo_idx = int(np.random.choice(tuning_splits.train_idx, size=1)[0])
demo_sample = ds_train[demo_idx]
demo_img_raw = demo_sample["images"].numpy()
demo_box = demo_sample["boxes"].numpy()
demo_img = apply_bbox_crop_optimized(demo_img_raw, demo_box, padding_ratio=0.15)
demo_img = _ensure_uint8_rgb(demo_img)
demo_img = _pad_to_square(demo_img)
augmentations = {
    "Original": None,
    "Horizontal Flip\n(p_hflip: 0-0.5)": A.HorizontalFlip(p=1.0),
    "Scale + Rotate\n(p_affine: 0-0.5\nscale: 0.05-0.2\nrotate: 5-30°)": A.Affine(
        scale=(0.85, 1.15),
        rotate=(-20, 20),
        shear={"x": 0, "y": 0},
        border_mode=cv2.BORDER_REFLECT_101,
        p=1.0,
    ),
    "Brightness+Contrast\n(p_brightness_contrast: 0-0.5)": A.ColorJitter(
        brightness=0.2,
        contrast=0.3,
        saturation=0.0,
        hue=0.0,
        p=1.0,
    ),
    "Hue+Saturation\n(p_hue_sat: 0-0.4)": A.ColorJitter(
        brightness=0.0,
        contrast=0.0,
        saturation=0.25,
        hue=0.08,
        p=1.0,
    ),
    "CLAHE\n(p_clahe: 0-0.4\nclip_limit: 1-4)": A.CLAHE(
        clip_limit=3.0,
        tile_grid_size=(8, 8),
        p=1.0,
    ),
    "Gaussian Blur\n(p_blur: 0-0.3)": A.GaussianBlur(blur_limit=(5, 7), p=1.0),
    "Gaussian Noise\n(p_noise: 0-0.3)": A.GaussNoise(std_range=(0.05, 0.1), p=1.0),
    "Cutout/Dropout\n(p_cutout: 0-0.5)": A.CoarseDropout(
        num_holes_range=(3, 5),
        hole_height_range=(0.08, 0.12),
        hole_width_range=(0.08, 0.12),
        p=1.0,
    ),
}

n_augs = len(augmentations)
n_cols = 3
n_rows = (n_augs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

for i, (name, aug) in enumerate(augmentations.items()):
    if aug is None:
        img_aug = demo_img.copy()
    else:
        result = aug(image=demo_img)
        img_aug = result["image"]
    if img_aug.dtype != np.uint8:
        img_aug = np.clip(img_aug, 0, 255).astype(np.uint8)

    axes[i].imshow(img_aug)
    axes[i].set_title(name, fontsize=10, fontweight="bold")
    axes[i].axis("off")
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Augmentation Search Space", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Image transformations include:
- Symmetric horizontal flips,
- Slight scale and rotations without shifting the bird,
- brightness and contrast tuning, and hue and saturation tuning,
- CLAHE (contrast limited adaptive histogram equalization) to bring out details in the bird,
- blur and noise to add imperfections and improve generalization, and
- cutout to force the model to look at multiple features.

All of these transformations preserve the original image shapes and colors, which is important for this task.

## Augmentation Search

The model is now tuned to find the best combination of augmentations for this task. Pruning is used to avoid spending unnecessary time training bad trials. A limited number of epochs are used for tuning the classifier head (2)and backbone (4), as it is assumed that promising augmentation combinations will show improvements by this time.

If the notebook is re-run, previously found optuna augmentation parameters are simply loaded:

In [ ]:
LOAD_SAVED_AUG_PARAMS = True

if LOAD_SAVED_AUG_PARAMS:
    AUG_PARAMS_FILE = RUNS_DIR / "best_augmentation_params.json"

    if AUG_PARAMS_FILE.exists():
        with open(AUG_PARAMS_FILE, "r") as f:
            aug_results = json.load(f)

        BEST_AUG_PARAMS = aug_results["best_params"]
        BEST_AUGMENTATION = "recipe"
        BEST_PREPROCESS = "bbox_crop"

        class MockStudy:
            def __init__(self, params, f1_score, trial_number):
                self.best_params = params
                self.best_value = f1_score
                self.best_trial = type("obj", (object,), {"number": trial_number})()

        study_aug = MockStudy(
            BEST_AUG_PARAMS,
            aug_results.get("best_f1", 0.0),
            aug_results.get("best_trial", -1),
        )

        print("=" * 60)
        print("LOADED SAVED AUGMENTATION PARAMS")
        print("=" * 60)
        print(f"Source: {AUG_PARAMS_FILE}")
        print(f"Best trial: #{aug_results.get('best_trial', 'N/A')}")
        print(
            f"Best F1: {aug_results.get('best_f1', 'N/A'):.4f}"
            if aug_results.get("best_f1")
            else "Best F1: N/A"
        )
        print()
        print("Augmentation params:")
        for k, v in BEST_AUG_PARAMS.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        print()
    else:
        print(f"Saved params file not found: {AUG_PARAMS_FILE}")
        print(
            "Set LOAD_SAVED_AUG_PARAMS = False and run the augmentation search instead."
        )
        LOAD_SAVED_AUG_PARAMS = False
else:
    print("LOAD_SAVED_AUG_PARAMS is False - run the augmentation search cell instead.")

Otherwise, the hyperparameter tuning is run:

In [ ]:
FAST_HEAD_EPOCHS = 2
FAST_FINETUNE_EPOCHS = 4
N_TRIALS_AUG = 20
BEST_PREPROCESS = "bbox_crop"

print("=" * 60)
print("AUGMENTATION SEARCH (Optuna recipe, Fast Mode)")
print("=" * 60)
print(f"Subset size: {len(tuning_splits.train_idx):,} train samples")
print(f"Epochs per trial: {FAST_HEAD_EPOCHS} head + {FAST_FINETUNE_EPOCHS} finetune")
print(f"Preprocessing: {BEST_PREPROCESS}")
print(f"Trials: {N_TRIALS_AUG}")
print()

study_aug = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=3,
        interval_steps=1,
    ),
    study_name="bird_classifier_aug_search",
)

_t0 = _time.time()
study_aug.optimize(aug_objective, n_trials=N_TRIALS_AUG, show_progress_bar=True)
print(f"\nCompleted in {(_time.time() - _t0)/60:.1f} min")

print("\n" + "=" * 60)
print("AUGMENTATION OPTUNA RESULTS")
print("=" * 60)
print(f"Best trial: #{study_aug.best_trial.number}")
print(f"Best val F1: {study_aug.best_value:.4f}")
print("\nBest augmentation params:")
for k, v in study_aug.best_params.items():
    print(f"  {k}: {v}")

aug_results = {
    "best_trial": study_aug.best_trial.number,
    "best_f1": study_aug.best_value,
    "best_params": study_aug.best_params,
}
aug_results_file = RUNS_DIR / "best_augmentation_params.json"
with open(aug_results_file, "w") as f:
    json.dump(aug_results, f, indent=2)
print(f"\nSaved best augmentation params to: {aug_results_file}")

BEST_AUGMENTATION = "recipe"
BEST_AUG_PARAMS = study_aug.best_params
print(
    f"Set BEST_AUGMENTATION='{BEST_AUGMENTATION}' and BEST_AUG_PARAMS for visualization"
)

The best augmentation parameters are visualized:

In [ ]:
weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1
best_aug = resolve_train_augmentation(
    BEST_AUGMENTATION if "BEST_AUGMENTATION" in globals() else "moderate",
    BEST_AUG_PARAMS if "BEST_AUG_PARAMS" in globals() else None,
)

print("Visualizing pipeline with:")
print(
    f"  Preprocessing: {BEST_PREPROCESS if 'BEST_PREPROCESS' in globals() else 'bbox_crop'}"
)
print(
    f"  Augmentation: {BEST_AUGMENTATION if 'BEST_AUGMENTATION' in globals() else 'moderate'}"
)
np.random.seed(SEED)
viz_indices = np.random.choice(tuning_splits.train_idx, size=5, replace=False)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(5, 4, figsize=(16, 20))
col_titles = [
    "Original",
    "Preprocessed (no aug)",
    "Augmented (run 1)",
    "Augmented (run 2)",
]

for i, idx in enumerate(viz_indices):
    idx = int(idx)
    sample = ds_train[idx]
    img_orig = sample["images"].numpy()
    label = int(sample["labels"].numpy().item())
    axes[i, 0].imshow(img_orig)
    axes[i, 0].set_title(f"Class {label}" if i == 0 else "", fontsize=10)
    axes[i, 0].axis("off")
    ds_no_aug = BirdDataset(
        ds_train,
        [idx],
        [label],
        weights=weights,
        preprocess_mode=(
            BEST_PREPROCESS if "BEST_PREPROCESS" in globals() else "bbox_crop"
        ),
        augmentation=None,
    )
    img_tensor = ds_no_aug[0][0]
    img_display = img_tensor.permute(1, 2, 0).numpy() * std + mean
    img_display = np.clip(img_display, 0, 1)
    axes[i, 1].imshow(img_display)
    axes[i, 1].axis("off")
    ds_with_aug = BirdDataset(
        ds_train,
        [idx],
        [label],
        weights=weights,
        preprocess_mode=(
            BEST_PREPROCESS if "BEST_PREPROCESS" in globals() else "bbox_crop"
        ),
        augmentation=best_aug,
    )
    for j in range(2):
        img_tensor = ds_with_aug[0][0]
        img_aug = img_tensor.permute(1, 2, 0).numpy() * std + mean
        img_aug = np.clip(img_aug, 0, 1)
        axes[i, 2 + j].imshow(img_aug)
        axes[i, 2 + j].axis("off")

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight="bold")

plt.suptitle("Best Preprocessing + Augmentation Pipeline", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Optuna's augmentation search found that geometric transformations were best, though no augmentations showed large increases for model F1 score (best validation F1 score was found to be 0.3065). As seen in the example image, augmentations include horizontal flips, rotations (up to 20 degrees), slight scale increase and decrease, adding noise, and adding cutout about 10% of the time. Very small light adjustments are seen, which is expected, as bird color is a key factor for distinguising bird species from each other.

## Model hyperparameter search

Now that the best preprocessing method has been found, model parameters such as learning rate, loss function, optimizer, regularization parameters, and leaerning rate scheduler are also tuned for the EfficientNet-B4 model on this dataset.

### Parameters Searched:
| Category | Parameters | Description |
|----------|------------|-------------|
| **Optimizer** | optimizer, momentum | AdamW vs SGD with momentum |
| **Learning Rates** | lr_head, lr_backbone, lr_head_finetune | Learning rates for head and backbone |
| **Regularization** | weight_decay, dropout, grad_clip_norm | Prevent overfitting |
| **Loss Function** | loss_fn, focal_gamma, label_smoothing | CrossEntropy, Focal, or LabelSmoothing |
| **Scheduler** | scheduler | None, Cosine, or OneCycle LR decay |

If the notebook is re-run, previously found optuna augmentation parameters are simply loaded:

In [ ]:
LOAD_SAVED_HPO_PARAMS = True

if LOAD_SAVED_HPO_PARAMS:
    HPO_PARAMS_FILE = RUNS_DIR / "best_hpo_params.json"

    if HPO_PARAMS_FILE.exists():
        with open(HPO_PARAMS_FILE, "r") as f:
            hpo_results = json.load(f)

        BEST_HPO_PARAMS = hpo_results["best_params"]
        BEST_AUGMENTATION = hpo_results.get("augmentation", "recipe")
        BEST_AUG_PARAMS = hpo_results.get("augmentation_params", {})
        BEST_PREPROCESS = hpo_results.get("preprocess", "bbox_crop")

        class MockHPOStudy:
            def __init__(self, params, f1_score, trial_number):
                self.best_params = params
                self.best_value = f1_score
                self.best_trial = type("obj", (object,), {"number": trial_number})()

        study = MockHPOStudy(
            BEST_HPO_PARAMS,
            hpo_results.get("best_f1", 0.0),
            hpo_results.get("best_trial", -1),
        )

        print("=" * 70)
        print("LOADED SAVED HPO PARAMS")
        print("=" * 70)
        print(f"Source: {HPO_PARAMS_FILE}")
        print(f"Best trial: #{hpo_results.get('best_trial', 'N/A')}")
        print(
            f"Best F1: {hpo_results.get('best_f1', 'N/A'):.4f}"
            if hpo_results.get("best_f1")
            else "Best F1: N/A"
        )
        print(f"Preprocessing: {BEST_PREPROCESS}")
        print(f"Augmentation: {BEST_AUGMENTATION}")
        print()
        print("HPO params:")
        for k, v in BEST_HPO_PARAMS.items():
            if isinstance(v, float):
                print(f"  {k}: {v:.6f}")
            else:
                print(f"  {k}: {v}")
        print()
    else:
        print(f"Saved params file not found: {HPO_PARAMS_FILE}")
        print("Set LOAD_SAVED_HPO_PARAMS = False and run the HPO search instead.")
        LOAD_SAVED_HPO_PARAMS = False
else:
    print("LOAD_SAVED_HPO_PARAMS is False - run the HPO search cell instead.")

Otherwise, the experiment is run:

In [ ]:
N_TRIALS = 20
OPTUNA_HEAD_EPOCHS = 2
OPTUNA_FINETUNE_EPOCHS = 4
BEST_AUGMENTATION = "recipe"
BEST_AUG_PARAMS = dict(study_aug.best_params)


print("=" * 70)
print("OPTUNA COMPREHENSIVE HYPERPARAMETER SEARCH")
print("=" * 70)
print(f"Trials: {N_TRIALS}")
print(
    f"Epochs per trial: {OPTUNA_HEAD_EPOCHS} head + {OPTUNA_FINETUNE_EPOCHS} finetune"
)
print(f"Preprocessing: {BEST_PREPROCESS}")
print(f"Augmentation: {BEST_AUGMENTATION}")
print(f"Subset size: {len(tuning_splits.train_idx):,} samples")
print()
print("Search space:")
print("  • Optimizer: [adamw, sgd]")
print("  • Loss: [cross_entropy, focal, label_smoothing]")
print("  • Scheduler: [none, cosine, onecycle]")
print("  • + learning rates, regularization, dropout")
print()

study = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=3,
        interval_steps=1,
    ),
    study_name="bird_classifier_hpo_v2",
)

objective = create_fast_objective(
    ds_train=ds_train,
    tuning_splits=tuning_splits,
    tuning_label_index=tuning_label_index,
    best_preprocess=BEST_PREPROCESS,
    best_augmentation=BEST_AUGMENTATION,
    best_augmentation_params=BEST_AUG_PARAMS,
    head_epochs=OPTUNA_HEAD_EPOCHS,
    finetune_epochs=OPTUNA_FINETUNE_EPOCHS,
)
_t0 = _time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
print(f"\nCompleted in {(_time.time() - _t0)/60:.1f} min")
print("\n" + "=" * 70)
print("OPTUNA RESULTS")
print("=" * 70)
print(f"Best trial: #{study.best_trial.number}")
print(f"Best val F1: {study.best_value:.4f}")
print("\nBest hyperparameters:")
for key, value in study.best_params.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.6f}")
    else:
        print(f"  {key}: {value}")

hpo_results = {
    "best_trial": study.best_trial.number,
    "best_f1": study.best_value,
    "best_params": study.best_params,
    "augmentation": BEST_AUGMENTATION,
    "augmentation_params": BEST_AUG_PARAMS,
    "preprocess": BEST_PREPROCESS,
}
hpo_results_file = RUNS_DIR / "best_hpo_params.json"
with open(hpo_results_file, "w") as f:
    json.dump(hpo_results, f, indent=2)
print(f"Saved best hyperparameters to: {hpo_results_file}")

From this search on the sample, the best optimizer was found to be AdamW, and cosine annealing what found to be the best learning rate scheduler. A low dropout rate (0.12) suggests that the model does best with high capacity. Also, the fact that focal loss function is not used suggests that class imbalance is not troublesome, as seen in the EDA notebook.

# Final training

After finding the best hyperparameters using a sample of the training data, the model is retrained all 95% of the training data, with 5% reserved for validation between epochs for early stopping and checkpoint selection. This maximizes the amount of samples available for the final learning setp before holdout data evaluation.

In [ ]:
if "study" in globals() and study.best_params:
    best_params = study.best_params
    print("Using Optuna best parameters:")
else:
    best_params = {
        "lr_head": 1e-3,
        "lr_backbone": 1e-5,
        "lr_head_finetune": 3e-4,
        "weight_decay": 1e-4,
        "label_smoothing": 0.1,
        "optimizer": "adamw",
        "loss_fn": "cross_entropy",
        "scheduler": "none",
        "dropout": 0.4,
        "momentum": 0.9,
        "grad_clip_norm": 1.0,
        "focal_gamma": 2.0,
    }
    print("Using default parameters (run Optuna search first for best results):")

for k, v in best_params.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

if "BEST_AUGMENTATION" in globals() and BEST_AUGMENTATION == "recipe":
    if "BEST_AUG_PARAMS" in globals() and BEST_AUG_PARAMS:
        FINAL_AUGMENTATION = "recipe"
        FINAL_AUG_PARAMS = BEST_AUG_PARAMS
        print("Using Optuna-tuned augmentation recipe")
    else:
        FINAL_AUGMENTATION = "moderate"
        FINAL_AUG_PARAMS = None
        print(
            "BEST_AUGMENTATION='recipe' but BEST_AUG_PARAMS missing - using 'moderate' preset"
        )
elif "BEST_AUGMENTATION" in globals():
    FINAL_AUGMENTATION = BEST_AUGMENTATION
    FINAL_AUG_PARAMS = None
    print(f"Using augmentation preset: '{FINAL_AUGMENTATION}'")
else:
    FINAL_AUGMENTATION = "moderate"
    FINAL_AUG_PARAMS = None
    print("No augmentation params found - using 'moderate' preset")

all_train_indices = np.concatenate([splits.train_idx, splits.val_idx, splits.test_idx])
all_train_labels = np.concatenate([splits.train_y, splits.val_y, splits.test_y])
train_idx, val_idx, train_y, val_y = train_test_split(
    all_train_indices,
    all_train_labels,
    test_size=0.05,
    stratify=all_train_labels,
    random_state=SEED,
)

full_splits = SplitIndices(
    train_idx=train_idx,
    train_y=train_y,
    val_idx=val_idx,
    val_y=val_y,
    test_idx=np.array([], dtype=int),
    test_y=np.array([], dtype=int),
)

print(f"\n{'=' * 60}")
print("FULL TRAINING DATA")
print(f"{'=' * 60}")
print(
    f"Original splits: train={len(splits.train_idx):,}, val={len(splits.val_idx):,}, test={len(splits.test_idx):,}"
)
print(f"Combined total:  {len(all_train_indices):,} samples")
print(
    f"Final splits:    train={len(full_splits.train_idx):,} (95%), val={len(full_splits.val_idx):,} (5%)"
)
print(f"Holdout (ds_val): {len(ds_val):,} samples (untouched for final evaluation)")

FINAL_HEAD_EPOCHS = 5
FINAL_FINETUNE_EPOCHS = 15

print(
    f"\nFinal training: {FINAL_HEAD_EPOCHS} head + {FINAL_FINETUNE_EPOCHS} finetune epochs"
)
print("\n" + "=" * 60)
print("FINAL MODEL TRAINING (100% of ds_train)")
print("=" * 60)

cfg = TrainConfig(
    run_name="final_full_data_model",
    preprocess_mode=BEST_PREPROCESS,
    augmentation=FINAL_AUGMENTATION,
    augmentation_params=FINAL_AUG_PARAMS,
    lr_head=best_params.get("lr_head", 1e-3),
    lr_backbone=best_params.get("lr_backbone", 1e-5),
    lr_head_finetune=best_params.get("lr_head_finetune", 3e-4),
    weight_decay=best_params.get("weight_decay", 1e-4),
    optimizer=best_params.get("optimizer", "adamw"),
    loss_fn=best_params.get("loss_fn", "cross_entropy"),
    focal_gamma=best_params.get("focal_gamma", 2.0),
    label_smoothing=best_params.get("label_smoothing", 0.0),
    scheduler=best_params.get("scheduler", "none"),
    dropout=best_params.get("dropout", 0.4),
    momentum=best_params.get("momentum", 0.9),
    grad_clip_norm=best_params.get("grad_clip_norm", 1.0),
    head_epochs=FINAL_HEAD_EPOCHS,
    finetune_epochs=FINAL_FINETUNE_EPOCHS,
    early_stop_patience=3,
    cache_dir=str(CACHE_DIR) if CACHE_DIR else None,
)

_t0 = _time.time()
final_model, final_history_df, final_summary, final_run_dir = train_two_stage(
    cfg=cfg,
    ds_train=ds_train,
    train_label_index=train_label_index,
    splits=full_splits,
    evaluate_test=False,
    evaluate_holdout=False,
    output_dir=RUNS_DIR,
)
print(f"\n Final training completed in {(_time.time() - _t0)/60:.1f} min")

print("\n" + "=" * 60)
print("FINAL MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"Best validation F1: {final_summary.get('val_best_f1', 0):.4f}")
print(f"Model saved to: {final_run_dir}")

plot_training_history(final_history_df, run_name="final_full_data_model")

The final trained model is still overfitting to the training data. The best F1 score was found on epoch 19, which shows that 20 epochs of training could have been increased, as the model consistenty improved with epoch number. The final validation accuracy has also increased to 80%.


If this notebook is being rerun, the final trained model is loaded instead:

In [ ]:
model_path = (
    "/content/drive/MyDrive/bird_classifier_runs/"
    "final_full_data_model/best_finetune.pt"
)

out = build_efficientnet_b4(num_classes=NUM_CLASSES, dropout=0.12)
final_model = out[0] if isinstance(out, tuple) else out

checkpoint = torch.load(model_path, map_location=DEVICE)
state_dict = (
    checkpoint.get("model_state_dict")
    or checkpoint.get("model_state")
    or checkpoint.get("state_dict")
    or checkpoint
)

PREFIX = "_orig_mod."
if all(k.startswith(PREFIX) for k in state_dict.keys()):
    state_dict = {k[len(PREFIX) :]: v for k, v in state_dict.items()}
else:
    state_dict = {
        k[len(PREFIX) :] if k.startswith(PREFIX) else k: v
        for k, v in state_dict.items()
    }

missing, unexpected = final_model.load_state_dict(state_dict, strict=True)

final_model = final_model.to(DEVICE)
final_model.eval()

print(f"Model loaded from: {model_path}")
print(f"  Epoch: {checkpoint.get('epoch', 'unknown')}")
val_f1 = checkpoint.get("val_f1", None)
if val_f1 is not None:
    print(f"  Val F1: {val_f1:.4f}")
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")

# Holdout Dataset Evaluation

The final holdout test set (ds_val from DeepLake) is used to evaluate the tuned classifier.
This section includes:
- Loading and preparing the holdout dataset
- Computing per-class accuracy and support statistics
- Confusion matrix analysis with species names
- Visual comparison of most confused class pairs  
- LIME explanations for correct and incorrect predictions

First, loading the validation data:

In [ ]:
print("=" * 70)
print("LOADING FINAL HOLDOUT TEST SET (ds_val)")
print("=" * 70)
ds_val = deeplake.load("hub://activeloop/nabirds-dataset-val", read_only=True)
print(f"Holdout samples (raw): {len(ds_val)}")
CLEAN_VAL_INDICES_PATHS = [
    Path(MODULE_PATH) / "clean_indices.npz",
    Path("clean_indices.npz"),
    DATA_ROOT / "clean_indices.npz",
]
_clean_val_npz = None
for p in CLEAN_VAL_INDICES_PATHS:
    if p.exists():
        _clean_val_npz = np.load(p)
        print(f"Loaded cleaned index file: {p}")
        break

CLEAN_VAL_INDICES = None
if _clean_val_npz is not None and "val_indices" in _clean_val_npz.files:
    CLEAN_VAL_INDICES = _clean_val_npz["val_indices"].astype(np.int64)
    print(
        f"Using cleaned val indices (duplicates dropped): {len(CLEAN_VAL_INDICES)} kept"
    )
print("\nBuilding val_label_index directly from ds_val (like fewshot.ipynb)...")
val_label_index = build_label_index(ds_val)
if CLEAN_VAL_INDICES is not None:
    allowed = set(map(int, CLEAN_VAL_INDICES))
    val_label_index = {
        int(class_id): np.array(
            [int(i) for i in indices if int(i) in allowed], dtype=np.int64
        )
        for class_id, indices in val_label_index.items()
    }
    total_clean = sum(len(v) for v in val_label_index.values())
    print(
        f"After cleaning: {total_clean} samples across {len(val_label_index)} classes"
    )

# verify class ID between train and val
print("\n" + "=" * 70)
print("CLASS ID CONSISTENCY CHECK")
print("=" * 70)

train_class_ids = set(train_label_index.keys())
val_class_ids = set(val_label_index.keys())

print(
    f"Training classes: {len(train_class_ids)} (range: {min(train_class_ids)}-{max(train_class_ids)})"
)
print(
    f"Validation classes: {len(val_class_ids)} (range: {min(val_class_ids)}-{max(val_class_ids)})"
)

common_classes = train_class_ids & val_class_ids
val_only = val_class_ids - train_class_ids

print(f"Common classes: {len(common_classes)}")
if val_only:
    print(f"{len(val_only)} val classes not in train (will skip)")

if "CLASS_ID_TO_IDX" not in globals() or len(CLASS_ID_TO_IDX) == 0:
    print("\nRecreating CLASS_ID_TO_IDX from train_label_index...")
    CLASS_IDS = sorted(train_label_index.keys())
    CLASS_ID_TO_IDX = {cid: i for i, cid in enumerate(CLASS_IDS)}
    NUM_CLASSES = len(CLASS_IDS)

IDX_TO_CLASS_ID = {idx: class_id for class_id, idx in CLASS_ID_TO_IDX.items()}

# build final test array with mapped label
final_test_indices = []
final_test_labels = []
final_test_class_ids = []

skipped_samples = 0
for class_id, indices in val_label_index.items():
    if class_id not in CLASS_ID_TO_IDX:
        skipped_samples += len(indices)
        continue
    model_idx = CLASS_ID_TO_IDX[class_id]
    final_test_indices.extend(indices.tolist())
    final_test_labels.extend([model_idx] * len(indices))
    final_test_class_ids.extend([class_id] * len(indices))

if skipped_samples > 0:
    print(f"Skipped {skipped_samples} samples from classes not in training")

final_test_indices = np.array(final_test_indices, dtype=np.int64)
final_test_labels = np.array(final_test_labels, dtype=np.int64)
final_test_class_ids = np.array(final_test_class_ids, dtype=np.int64)

val_num_classes = len(set(final_test_labels))

print(f"\nFinal holdout test set prepared:")
print(f"   Total samples: {len(final_test_labels)}")
print(f"   Unique classes: {val_num_classes}")
print(f"   Model index range: [{final_test_labels.min()}, {final_test_labels.max()}]")

Next, the final model is evaluated on the unseen test data:

In [ ]:
# Fix tqdm cleanup issue in notebooks
import gc
from tqdm.auto import tqdm as _tqdm_auto

# Close any lingering tqdm instances to prevent AttributeError on garbage collection
try:
    for obj in gc.get_objects():
        if isinstance(obj, _tqdm_auto):
            try:
                obj.close()
            except Exception:
                pass
except Exception:
    pass
gc.collect()

In [ ]:
if "final_model" in globals() and final_model is not None:
    model = final_model
    print("Using final_model from previous cell")
    holdout_preprocess_mode = (
        BEST_PREPROCESS if "BEST_PREPROCESS" in globals() else "bbox_crop"
    )
    print(f"Using preprocessing mode: {holdout_preprocess_mode}")
else:
    print("final_model not found in memory, loading from disk...")

    BEST_MODEL_CANDIDATES = [
        RUNS_DIR / "final_full_data_model" / "best_finetune.pt",
        RUNS_DIR / "final_best_model" / "best_finetune.pt",
        RUNS_DIR / "exp2a_bbox_finetune" / "best_finetune.pt",
        RUNS_DIR / "exp2a_native_finetune" / "best_finetune.pt",
    ]

    best_model_path = next((p for p in BEST_MODEL_CANDIDATES if p.exists()), None)
    if best_model_path is None:
        raise FileNotFoundError(
            "No trained model checkpoint found. Please run the training cells first!\n"
            f"Searched in: {RUNS_DIR}"
        )

    print(f"Loading best model from: {best_model_path}")
    model, _ = build_efficientnet_b4(num_classes=NUM_CLASSES)
    checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)
    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        model.load_state_dict(checkpoint["model_state"])
    else:
        model.load_state_dict(checkpoint)
    model = model.to(DEVICE)

    _model_config_path = best_model_path.parent / "config.json"
    if _model_config_path.exists():
        with open(_model_config_path) as f:
            _model_config = json.load(f)
        holdout_preprocess_mode = _model_config.get("preprocess_mode", "bbox_crop")
    else:
        holdout_preprocess_mode = (
            BEST_PREPROCESS if "BEST_PREPROCESS" in globals() else "bbox_crop"
        )

model = model.to(DEVICE)
model.eval()
print(f"Model ready ({sum(p.numel() for p in model.parameters()):,} parameters)")

weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1

# Create holdout dataset
holdout_dataset = BirdDataset(
    ds_val,
    final_test_indices,
    final_test_labels,
    weights=weights,
    preprocess_mode=holdout_preprocess_mode,
    augmentation=None,
    cache_dir=None,
)

holdout_loader = make_dataloader(
    holdout_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"Evaluating on holdout set ({len(holdout_dataset)} samples)...")

# Run evaluation
criterion = nn.CrossEntropyLoss()
holdout_metrics, holdout_true, holdout_preds, holdout_indices = evaluate_with_preds(
    model, holdout_loader, DEVICE, criterion
)
holdout_precision = precision_score(
    holdout_true, holdout_preds, average="macro", zero_division=0
)
holdout_recall = recall_score(
    holdout_true, holdout_preds, average="macro", zero_division=0
)
holdout_predictions = np.array(holdout_preds)
holdout_true_labels = np.array(holdout_true)
print("\n" + "=" * 80)
print("HOLDOUT RESULTS (ds_val)")
print("=" * 80)
print(f"  Accuracy:       {holdout_metrics['acc']*100:.2f}%")
print(f"  Top-5 Accuracy: {holdout_metrics['top5_acc']*100:.2f}%")
print(f"  Precision:      {holdout_precision*100:.2f}%")
print(f"  Recall:         {holdout_recall*100:.2f}%")
print(f"  F1 Score:       {holdout_metrics['f1']*100:.2f}%")
print(f"  Loss:           {holdout_metrics['loss']:.4f}")
print("=" * 80)

For the final model, accuracy, precision, and recall are all between 75-77%. This is higher than the 71% holdout accuracy found using the few shot approach.

Now, examining per-class accuracy:

In [ ]:
holdout_class_stats = defaultdict(lambda: {"correct": 0, "total": 0, "predictions": []})
for true_label, pred_label in zip(holdout_true_labels, holdout_predictions):
    true_label = int(true_label)
    pred_label = int(pred_label)
    holdout_class_stats[true_label]["total"] += 1
    holdout_class_stats[true_label]["predictions"].append(pred_label)
    if true_label == pred_label:
        holdout_class_stats[true_label]["correct"] += 1

# Calculate accuracy for each class
for model_idx in holdout_class_stats:
    stats = holdout_class_stats[model_idx]
    stats["accuracy"] = stats["correct"] / stats["total"] if stats["total"] > 0 else 0.0
    stats["support_size"] = stats["total"]
try:
    class_names = ds_val.labels.info.class_names
except:
    try:
        class_names = ds_train.labels.info.class_names
    except:
        class_names = [f"Class_{i}" for i in range(max(CLASS_IDS) + 1)]
IDX_TO_CLASS_ID = {idx: class_id for class_id, idx in CLASS_ID_TO_IDX.items()}
holdout_accuracies = np.array(
    [holdout_class_stats[c]["accuracy"] for c in holdout_class_stats.keys()]
)

print("=" * 70)
print("PER-CLASS ACCURACY STATISTICS (HOLDOUT)")
print("=" * 70)
print(f"Total classes evaluated: {len(holdout_class_stats)}")
print(f"Mean per-class accuracy: {np.mean(holdout_accuracies)*100:.2f}%")
print(f"Median per-class accuracy: {np.median(holdout_accuracies)*100:.2f}%")
print(f"Std per-class accuracy: {np.std(holdout_accuracies)*100:.2f}%")

# Classes meeting 70% target
classes_above_70 = sum(1 for acc in holdout_accuracies if acc >= 0.70)
print(
    f"\nClasses ≥70% accuracy: {classes_above_70}/{len(holdout_accuracies)} ({classes_above_70/len(holdout_accuracies)*100:.1f}%)"
)

holdout_sample_info = []
for i, (true_lbl, pred_lbl) in enumerate(zip(holdout_true_labels, holdout_predictions)):
    holdout_sample_info.append(
        {
            "idx": i,
            "ds_idx": int(final_test_indices[i]),
            "true": int(true_lbl),
            "pred": int(pred_lbl),
            "correct": int(true_lbl) == int(pred_lbl),
        }
    )

# Get worst and best classes
worst_5 = sorted(holdout_class_stats.items(), key=lambda x: x[1]["accuracy"])[:5]
best_5 = sorted(
    holdout_class_stats.items(), key=lambda x: x[1]["accuracy"], reverse=True
)[:5]

all_accs = list(holdout_accuracies)
print("\n" + "-" * 70)
print("HOLDOUT SUMMARY")
print("-" * 70)
print(f"  Overall Accuracy:          {holdout_metrics['acc']*100:.2f}%")
print(f"  Mean Per-Class Accuracy:   {np.mean(all_accs)*100:.2f}%")
print(f"  Median Per-Class Accuracy: {np.median(all_accs)*100:.2f}%")
print(f"  Classes with 0% accuracy:  {sum(1 for a in all_accs if a == 0)}")
print(f"  Classes with 100% accuracy:{sum(1 for a in all_accs if a == 1.0)}")
print("=" * 70)

Final test accuracy (77%) is slightly lower than final validation accuracy (80%), and does meet the task target of 70%. No classes are seen with 0% accuracy, and the median per-class accuracy (79%), which indicates that the model is generalizing well.

The best and worst classes are visualized:

In [ ]:
def get_class_name(
    model_idx: int,
    idx_to_class_id: Dict[int, int],
    class_names: List[str],
) -> str:
    """Get class name from model index.

    Args:
        model_idx: Model's internal class index.
        idx_to_class_id: Dict mapping model index to original class ID.
        class_names: List of class names indexed by class ID.

    Returns:
        Class name string.
    """
    class_id = idx_to_class_id.get(model_idx, model_idx)
    if class_id < len(class_names):
        return class_names[class_id]
    return f"Class_{class_id}"


print("=" * 80)
print("WORST 5 CLASSES - Sample Images")
print("=" * 80)

n_samples = 4

for model_idx, stats in worst_5:
    class_id = IDX_TO_CLASS_ID.get(model_idx, model_idx)
    name = get_class_name(model_idx)
    short_name = name.split("/")[-1] if "/" in name else name
    acc = stats["accuracy"]
    mistakes = [
        s for s in holdout_sample_info if s["true"] == model_idx and not s["correct"]
    ]
    train_indices = list(train_label_index.get(class_id, []))

    fig, axes = plt.subplots(1, n_samples, figsize=(4 * n_samples, 4))

    # Column 1: Reference from training set
    if train_indices:
        img = ds_train["images"][int(train_indices[0])].numpy()
        axes[0].imshow(img)
        axes[0].set_title(
            f"REFERENCE (train)\n{short_name[:25]}",
            fontsize=10,
            fontweight="bold",
            color="blue",
        )
        for spine in axes[0].spines.values():
            spine.set_edgecolor("blue")
            spine.set_linewidth(2)
    else:
        axes[0].set_title("No train ref", fontsize=10)
    axes[0].axis("off")

    # Columns 2+: Misclassified examples from holdout
    for i in range(1, n_samples):
        ax = axes[i]
        if i - 1 < len(mistakes):
            ds_idx = mistakes[i - 1]["ds_idx"]
            pred_model_idx = mistakes[i - 1]["pred"]
            pred_class_id = IDX_TO_CLASS_ID.get(pred_model_idx, pred_model_idx)
            pred_name = get_class_name(pred_model_idx)
            pred_short = pred_name.split("/")[-1] if "/" in pred_name else pred_name

            img = ds_val["images"][ds_idx].numpy()
            ax.imshow(img)
            ax.set_title(
                f"MISCLASSIFIED\nPred: {pred_short[:20]}", fontsize=9, color="red"
            )
            for spine in ax.spines.values():
                spine.set_edgecolor("red")
                spine.set_linewidth(2)
        else:
            ax.set_visible(False)
        ax.axis("off")

    plt.suptitle(
        f"WORST: {short_name} - {acc*100:.1f}% ({stats['correct']}/{stats['total']})",
        fontsize=12,
        fontweight="bold",
        color="red",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 80)
print("BEST 5 CLASSES - Sample Images")
print("=" * 80)

for model_idx, stats in best_5:
    class_id = IDX_TO_CLASS_ID.get(model_idx, model_idx)
    name = get_class_name(model_idx)
    short_name = name.split("/")[-1] if "/" in name else name
    acc = stats["accuracy"]
    correct_samples = [
        s for s in holdout_sample_info if s["true"] == model_idx and s["correct"]
    ]
    train_indices = list(train_label_index.get(class_id, []))

    fig, axes = plt.subplots(1, n_samples, figsize=(4 * n_samples, 4))

    # Column 1: Reference from training set
    if train_indices:
        img = ds_train["images"][int(train_indices[0])].numpy()
        axes[0].imshow(img)
        axes[0].set_title(
            f"REFERENCE (train)\n{short_name[:25]}",
            fontsize=10,
            fontweight="bold",
            color="blue",
        )
        for spine in axes[0].spines.values():
            spine.set_edgecolor("blue")
            spine.set_linewidth(2)
    else:
        axes[0].set_title("No train ref", fontsize=10)
    axes[0].axis("off")

    # Columns 2+: Correctly classified examples from holdout
    for i in range(1, n_samples):
        ax = axes[i]
        if i - 1 < len(correct_samples):
            ds_idx = correct_samples[i - 1]["ds_idx"]
            img = ds_val["images"][ds_idx].numpy()
            ax.imshow(img)
            ax.set_title(f"CORRECT ✓", fontsize=9, color="green")
            for spine in ax.spines.values():
                spine.set_edgecolor("green")
                spine.set_linewidth(2)
        else:
            ax.set_visible(False)
        ax.axis("off")

    plt.suptitle(
        f"BEST: {short_name} - {acc*100:.1f}% ({stats['correct']}/{stats['total']})",
        fontsize=12,
        fontweight="bold",
        color="green",
        y=1.02,
    )
    plt.tight_layout()
    plt.show()

Worst performing classes include confusion between different subspecies, which similar to the few shot model findings. For example, the dark-eyed junco species are easily confused with each other. The model also seems to have trouble recognizing same species birds with different poses (flying vs swimming or perched, for example). An error in recognition or labeling seems to have occurred for the Eclipse Male) class, where the model's correct predictions are labeled as incorrect. Best performing classes show birds (often in the same position) with various backgrounds, which indicates that the model is not using the image background to classify the species too much.

Examining the distribution of class accuracy performance, as this is the key comparison metric between the few shot approach and the fine-tuning approach:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 1. Histogram of per-class accuracies
ax1 = axes[0]
ax1.hist(holdout_accuracies, bins=25, edgecolor="black", alpha=0.7, color="steelblue")
ax1.axvline(
    np.mean(holdout_accuracies),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(holdout_accuracies)*100:.1f}%",
)
ax1.axvline(
    np.median(holdout_accuracies),
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {np.median(holdout_accuracies)*100:.1f}%",
)
ax1.axvline(0.7, color="green", linestyle=":", linewidth=2, label="Target: 70%")
ax1.set_xlabel("Per-Class Accuracy", fontsize=12)
ax1.set_ylabel("Number of Classes", fontsize=12)
ax1.set_title("Distribution of Per-Class Accuracy (Holdout)", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Accuracy vs Support Size scatter
ax2 = axes[1]
support_sizes = [
    holdout_class_stats[c]["support_size"] for c in holdout_class_stats.keys()
]
class_accs = [holdout_class_stats[c]["accuracy"] for c in holdout_class_stats.keys()]
ax2.scatter(support_sizes, class_accs, alpha=0.5, c="steelblue", edgecolor="white")
ax2.set_xlabel("Support Set Size (samples per class)", fontsize=12)
ax2.set_ylabel("Per-Class Accuracy", fontsize=12)
ax2.set_title("Accuracy vs Support Size (Holdout)", fontsize=14)
ax2.axhline(0.7, color="green", linestyle="--", alpha=0.7, label="Target 70%")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.suptitle(
    "Holdout Test Set: Per-Class Performance Analysis",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

The distribution of per-class accuracy is slightly skewed to the left, where most classes have accuracy over 79%, and some have quite low accuracy. A slight trend is seen when examining accuracy and support size per cless, where classes with larger sample size also have higher accuracy.

Looking at a list of the most confused pairs:

In [ ]:
confused_pairs = compute_confused_pairs(
    holdout_true_labels, holdout_predictions, top_n=20
)
holdout_class_ids = set(int(c) for c in holdout_true_labels)

print("=" * 80)
print("TOP 15 CONFUSED CLASS PAIRS")
print("=" * 80)
for i, pair in enumerate(confused_pairs[:15]):
    true_model_idx = pair["true_class"]
    pred_model_idx = pair["pred_class"]
    true_class_id = IDX_TO_CLASS_ID.get(true_model_idx, true_model_idx)
    pred_class_id = IDX_TO_CLASS_ID.get(pred_model_idx, pred_model_idx)

    true_name = (
        class_names[true_class_id]
        if true_class_id < len(class_names)
        else f"Class_{true_class_id}"
    )
    pred_name = (
        class_names[pred_class_id]
        if pred_class_id < len(class_names)
        else f"Class_{pred_class_id}"
    )

    true_short = true_name.split("/")[-1] if "/" in true_name else true_name
    pred_short = pred_name.split("/")[-1] if "/" in pred_name else pred_name

    print(f"{i+1:2}. {pair['count']:3} times: {true_short[:30]} → {pred_short[:30]}")

Easliy confused classes are mostly bird subspecies. This was also seen in the few shot approach, and in the EDA notebook similarity section, where Chickadee species and Hummingbird species look very similar, and are easily confused with each other.

Visualizing these:

In [ ]:
print("=" * 100)
print("MOST CONFUSED CLASS PAIRS - Visual Comparison")
print("=" * 100)

for pair_idx, pair in enumerate(confused_pairs[:5]):
    true_model_idx = int(pair["true_class"])
    pred_model_idx = int(pair["pred_class"])
    true_class_id = IDX_TO_CLASS_ID.get(true_model_idx, true_model_idx)
    pred_class_id = IDX_TO_CLASS_ID.get(pred_model_idx, pred_model_idx)

    true_name = (
        class_names[true_class_id]
        if true_class_id < len(class_names)
        else f"Class_{true_class_id}"
    )
    pred_name = (
        class_names[pred_class_id]
        if pred_class_id < len(class_names)
        else f"Class_{pred_class_id}"
    )

    true_short = true_name.split("/")[-1] if "/" in true_name else true_name
    pred_short = pred_name.split("/")[-1] if "/" in pred_name else pred_name

    print(f"\n{'='*100}")
    print(f"Pair {pair_idx+1}: {pair['count']} misclassifications")
    print(f"  TRUE CLASS:      {true_name} (ID: {true_class_id})")
    print(f"  PREDICTED AS:    {pred_name} (ID: {pred_class_id})")
    print(f"{'='*100}")
    confusion_examples = []
    for idx in range(len(holdout_true_labels)):
        if (
            holdout_true_labels[idx] == true_model_idx
            and holdout_predictions[idx] == pred_model_idx
        ):
            confusion_examples.append(final_test_indices[idx])
    train_true_cls_indices = list(train_label_index.get(true_class_id, []))
    train_pred_cls_indices = list(train_label_index.get(pred_class_id, []))

    n_confused = min(3, len(confusion_examples))
    n_cols = 2 + n_confused

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5))

    # Column 1: True class reference (from training set)
    if train_true_cls_indices:
        img = ds_train["images"][int(train_true_cls_indices[0])].numpy()
        axes[0].imshow(img)
        axes[0].set_title(
            f"TRUE CLASS (train ref)\n{true_short[:30]}",
            fontsize=11,
            fontweight="bold",
            color="green",
        )
        axes[0].set_xlabel(f"(ID: {true_class_id})", fontsize=9)
        for spine in axes[0].spines.values():
            spine.set_edgecolor("green")
            spine.set_linewidth(3)
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    # Column 2: Predicted class reference (from training set)
    if train_pred_cls_indices:
        img = ds_train["images"][int(train_pred_cls_indices[0])].numpy()
        axes[1].imshow(img)
        axes[1].set_title(
            f"CONFUSED WITH (train ref)\n{pred_short[:30]}",
            fontsize=11,
            fontweight="bold",
            color="red",
        )
        axes[1].set_xlabel(f"(ID: {pred_class_id})", fontsize=9)
        for spine in axes[1].spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(3)
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    # Columns 3+: Misclassified examples from holdout
    for i in range(n_confused):
        ax = axes[2 + i]
        if i < len(confusion_examples):
            img = ds_val["images"][int(confusion_examples[i])].numpy()
            ax.imshow(img)
            ax.set_title(
                f"MISCLASSIFIED #{i+1}\n(from holdout)", fontsize=10, color="orange"
            )
            ax.set_xlabel(f"True: {true_class_id} → Pred: {pred_class_id}", fontsize=8)
            for spine in ax.spines.values():
                spine.set_edgecolor("orange")
                spine.set_linewidth(2)
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()

indeed, these highly confused classes are all subspecies of each other. Birds of both species look very similar to each other in terms of body size, plumage, beak shapr, colors, and even background sometimes.

In terms of a confusion matrix:

In [ ]:
n_pairs_to_show = 10
pair_classes = []
for pair in confused_pairs[:n_pairs_to_show]:
    pair_classes.append((pair["true_class"], pair["pred_class"]))
unique_classes = sorted(set(c for pair in pair_classes for c in pair))
valid_classes = [c for c in unique_classes if c in holdout_class_ids]
n_cls = len(valid_classes)
cm_focused = np.zeros((n_cls, n_cls), dtype=int)
class_to_idx = {c: i for i, c in enumerate(valid_classes)}

for true_lbl, pred_lbl in zip(holdout_true_labels, holdout_predictions):
    true_lbl, pred_lbl = int(true_lbl), int(pred_lbl)
    if true_lbl in class_to_idx and pred_lbl in class_to_idx:
        cm_focused[class_to_idx[true_lbl], class_to_idx[pred_lbl]] += 1

short_names = []
for model_idx in valid_classes:
    class_id = IDX_TO_CLASS_ID.get(int(model_idx), int(model_idx))
    name = class_names[class_id] if class_id < len(class_names) else f"Class_{class_id}"
    short_name = name.split("/")[-1] if "/" in name else name
    short_names.append(short_name[:28])

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_focused,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=short_names,
    yticklabels=short_names,
    cbar_kws={"label": "Count"},
)
plt.xlabel("Predicted Species", fontsize=12)
plt.ylabel("True Species", fontsize=12)
plt.title(f"Top {n_pairs_to_show} Most Confused Pairs", fontsize=14)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

The same trend of confusing subspecies is seen here, and one species is generally not commonly confused with multiple species.

Now, examining the LIME explanations for the model's best and worst predictions:

In [ ]:
def predict_fn_lime(images):
    """LIME prediction function returning class probabilities."""
    probs = []
    model.eval()
    with torch.no_grad():
        for img in images:
            if img.dtype != np.uint8:
                img = np.clip(img, 0, 255).astype(np.uint8)
            pil_img = Image.fromarray(img)
            tensor = preprocess(pil_img).unsqueeze(0).to(DEVICE)
            logits = model(tensor)
            prob = F.softmax(logits, dim=1).cpu().numpy()[0]
            probs.append(prob)
    return np.array(probs)

In [ ]:
print("=" * 80)
print("LIME EXPLANATIONS: BEST vs WORST PREDICTIONS (HOLDOUT)")
print("=" * 80)

print("\n Computing prediction confidences...")
model.eval()
weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1
preprocess = weights.transforms()

all_confidences = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(
        tqdm(holdout_loader, desc="Getting confidences")
    ):
        images = images.to(DEVICE)
        logits = model(images)
        probs = F.softmax(logits, dim=1)
        max_probs, preds = probs.max(dim=1)

        for i in range(len(labels)):
            sample_idx = batch_idx * holdout_loader.batch_size + i
            if sample_idx < len(holdout_true_labels):
                all_confidences.append(
                    {
                        "sample_idx": sample_idx,
                        "ds_idx": int(final_test_indices[sample_idx]),
                        "true_label": int(holdout_true_labels[sample_idx]),
                        "pred_label": int(holdout_predictions[sample_idx]),
                        "confidence": float(max_probs[i].cpu()),
                        "correct": int(holdout_true_labels[sample_idx])
                        == int(holdout_predictions[sample_idx]),
                    }
                )

# Sort by confidence and select best predictions
all_confidences.sort(key=lambda x: x["confidence"], reverse=True)
best_predictions = [c for c in all_confidences if c["correct"]][:5]
print(f"\n BEST PREDICTIONS (highest confidence, correct):")
for i, pred in enumerate(best_predictions):
    true_name = get_class_name(pred["true_label"])
    short_name = true_name.split("/")[-1] if "/" in true_name else true_name
    print(f"   {i+1}. {short_name[:40]} - Confidence: {pred['confidence']*100:.1f}%")

# Select worst predictions
worst_predictions = [c for c in all_confidences if not c["correct"]][:5]
if len(worst_predictions) < 5:
    lowest_conf_correct = [c for c in reversed(all_confidences) if c["correct"]]
    worst_predictions.extend(lowest_conf_correct[: 5 - len(worst_predictions)])

print(f"\n WORST PREDICTIONS (incorrect or low confidence):")
for i, pred in enumerate(worst_predictions):
    true_name = get_class_name(pred["true_label"])
    pred_name = get_class_name(pred["pred_label"])
    true_short = true_name.split("/")[-1] if "/" in true_name else true_name
    pred_short = pred_name.split("/")[-1] if "/" in pred_name else pred_name
    status = "WRONG" if not pred["correct"] else "low conf"
    print(
        f"   {i+1}. True: {true_short[:25]} → Pred: {pred_short[:25]} ({pred['confidence']*100:.1f}%) [{status}]"
    )

# LIME setup
holdout_bbox_padding_ratio = 0.15
holdout_pad_to_square = True
if "_model_config" in globals() and _model_config is not None:
    holdout_bbox_padding_ratio = _model_config.get(
        "bbox_padding_ratio", holdout_bbox_padding_ratio
    )
    holdout_pad_to_square = _model_config.get("pad_to_square", holdout_pad_to_square)

explainer = lime_image.LimeImageExplainer()

# LIME for BEST predictions
print("\n" + "=" * 80)
print("LIME EXPLANATIONS: BEST PREDICTIONS")
print("=" * 80)

n_best = len(best_predictions)
if n_best > 0:
    fig, axes = plt.subplots(n_best, 4, figsize=(20, 5 * n_best))
    if n_best == 1:
        axes = axes.reshape(1, -1)

    for row, pred_info in enumerate(best_predictions):
        ds_idx = pred_info["ds_idx"]
        true_label = pred_info["true_label"]
        pred_label = pred_info["pred_label"]
        confidence = pred_info["confidence"]

        img_for_lime = prepare_lime_image(
            ds_val,
            ds_idx,
            holdout_preprocess_mode,
            holdout_bbox_padding_ratio,
            holdout_pad_to_square,
        )

        try:
            explanation = explainer.explain_instance(
                img_for_lime,
                predict_fn_lime,
                top_labels=10,
                hide_color=0,
                num_samples=500,
                random_seed=42,
            )

            species_name = get_class_name(true_label)
            short_name = (
                species_name.split("/")[-1] if "/" in species_name else species_name
            )
            available_labels = list(explanation.local_exp.keys())

            # Column 1: Original image
            axes[row, 0].imshow(img_for_lime)
            axes[row, 0].set_title(
                f"BEST #{row+1}\n{short_name[:30]}\nConf: {confidence*100:.1f}%",
                fontsize=10,
                fontweight="bold",
                color="green",
            )
            axes[row, 0].axis("off")

            # Column 2: Positive features
            if pred_label in available_labels:
                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )
                axes[row, 1].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 1].set_title(
                    "Positive Features\n(supporting prediction)", fontsize=9
                )
            else:
                axes[row, 1].imshow(img_for_lime)
                axes[row, 1].set_title("No explanation", fontsize=9)
            axes[row, 1].axis("off")

            # Column 3: All features
            if pred_label in available_labels:
                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=False, num_features=10, hide_rest=False
                )
                axes[row, 2].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 2].set_title("All Features\n(green=+, red=-)", fontsize=9)
            else:
                axes[row, 2].imshow(img_for_lime)
                axes[row, 2].set_title("No explanation", fontsize=9)
            axes[row, 2].axis("off")

            # Column 4: Key regions only
            if pred_label in available_labels:
                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=True
                )
                axes[row, 3].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 3].set_title("Key Regions Only\n(rest hidden)", fontsize=9)
            else:
                axes[row, 3].imshow(img_for_lime)
                axes[row, 3].set_title("No explanation", fontsize=9)
            axes[row, 3].axis("off")

            print(f"Best prediction #{row+1}: {short_name} ({confidence*100:.1f}%)")

        except Exception as e:
            print(f"LIME failed for best #{row+1}: {e}")
            for col in range(4):
                axes[row, col].imshow(img_for_lime)
                axes[row, col].set_title("Error", fontsize=9)
                axes[row, col].axis("off")

    plt.suptitle(
        "LIME Explanations: BEST Predictions (High Confidence, Correct)",
        fontsize=14,
        fontweight="bold",
        y=1.02,
        color="green",
    )
    plt.tight_layout()
    plt.show()

# LIME for worst predictions
print("\n" + "=" * 80)
print("LIME EXPLANATIONS: WORST PREDICTIONS")
print("=" * 80)

n_worst = len(worst_predictions)
if n_worst > 0:
    fig, axes = plt.subplots(n_worst, 4, figsize=(20, 5 * n_worst))
    if n_worst == 1:
        axes = axes.reshape(1, -1)

    for row, pred_info in enumerate(worst_predictions):
        ds_idx = pred_info["ds_idx"]
        true_label = pred_info["true_label"]
        pred_label = pred_info["pred_label"]
        confidence = pred_info["confidence"]
        is_correct = pred_info["correct"]

        img_for_lime = prepare_lime_image(
            ds_val,
            ds_idx,
            holdout_preprocess_mode,
            holdout_bbox_padding_ratio,
            holdout_pad_to_square,
        )

        try:
            explanation = explainer.explain_instance(
                img_for_lime,
                predict_fn_lime,
                top_labels=10,
                hide_color=0,
                num_samples=500,
                random_seed=42,
            )

            true_name = get_class_name(true_label)
            pred_name = get_class_name(pred_label)
            true_short = true_name.split("/")[-1] if "/" in true_name else true_name
            pred_short = pred_name.split("/")[-1] if "/" in pred_name else pred_name
            available_labels = list(explanation.local_exp.keys())

            # Column 1: Original with error info
            axes[row, 0].imshow(img_for_lime)
            status = "WRONG" if not is_correct else "Low Conf"
            axes[row, 0].set_title(
                f"WORST #{row+1} [{status}]\nTrue: {true_short[:20]}\nPred: {pred_short[:20]} ({confidence*100:.1f}%)",
                fontsize=9,
                fontweight="bold",
                color="red",
            )
            axes[row, 0].axis("off")

            # Column 2: Positive features (same as best)
            if pred_label in available_labels:
                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=False
                )
                axes[row, 1].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 1].set_title(
                    "Positive Features\n(supporting prediction)", fontsize=9
                )
            else:
                axes[row, 1].imshow(img_for_lime)
                axes[row, 1].set_title("No explanation", fontsize=9)
            axes[row, 1].axis("off")

            # Column 3: All features (same as best)
            if pred_label in available_labels:
                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=False, num_features=10, hide_rest=False
                )
                axes[row, 2].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 2].set_title("All Features\n(green=+, red=-)", fontsize=9)
            else:
                axes[row, 2].imshow(img_for_lime)
                axes[row, 2].set_title("No explanation", fontsize=9)
            axes[row, 2].axis("off")

            # Column 4: Key regions only (same as best)
            if pred_label in available_labels:
                temp, mask = explanation.get_image_and_mask(
                    pred_label, positive_only=True, num_features=5, hide_rest=True
                )
                axes[row, 3].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 3].set_title("Key Regions Only\n(rest hidden)", fontsize=9)
            else:
                axes[row, 3].imshow(img_for_lime)
                axes[row, 3].set_title("No explanation", fontsize=9)
            axes[row, 3].axis("off")

            print(
                f"Worst prediction #{row+1}: True={true_short[:20]}, Pred={pred_short[:20]} ({confidence*100:.1f}%)"
            )

        except Exception as e:
            print(f"LIME failed for worst #{row+1}: {e}")
            for col in range(4):
                axes[row, col].imshow(img_for_lime)
                axes[row, col].set_title("Error", fontsize=9)
                axes[row, col].axis("off")

    plt.suptitle(
        "LIME Explanations: WORST Predictions (Incorrect or Low Confidence)",
        fontsize=14,
        fontweight="bold",
        y=1.02,
        color="red",
    )
    plt.tight_layout()
    plt.show()

From the LIME explanations, it is possible to deduce that the trained EfficienNet-B4 model is properly identifying parts of the birds. Some background areas are used to help or hurt predictions.  For best predictions, key image areas that help to correctly classify the image include the bird’s head and sometimes feet. No large difference is seen between best and worst predicted LIME images, and a potential reason for these bad predictions is high inter-class similarity between species, as previously noted.



# Conclusion

In conclusion, this notebook illustrates a transfer learning and fine tuning approach to classify images from the NABirds dataset, which contains 555 North American bird species. The EfficientNet-B4 model was chosen to match results from the few shot learning approach. The model head was first  tuned, and bird bounding box preprocessing was found to be the best initial processing method (test accuracy of 73%). The model backbone was also fine tuned, which increased test accuracy to 83%. To find optimal image augmentations, optuna was used to fine tune the model with various color and geometric augmentations. This showed that very light color augmentations with slight occulusions and geometric changes such as scale, rotation, and horizontal flip are best for this dataset. Indeed, it was found in EDA that color preservation is essential, as minute differences in color can indicate a new bird species. Another round of hyperparameter tuning was then performed to find that the best optimizer was AdamW, with slight dropout (0.12), cross entropy loss, and a cosine learning rate scheduler. The model was then trained one last time on the full training data before evaluation on the holdout dataset.

As compared to the few shot learning approach, the fine tuning approach takes longer, but showed a 5% increase in accuracy with higher recall, F1 score, and precision. Similar issues in both cases were found to confuse the model, such as slight differences in sub species leading to inaccurate predictions. LIME images also showed that worst predicted samples were subspecies of each other. From these images, the model seems to be correctly identifying the bird and using some of the bird's background, and often the bird's head to help predict the species.


Further work:
- Enabling hierarchical classification to reduce subspecies confusion could help to improve accuracy. For example, classifying a general species first and then classifying the sub-species.
- Fine tuning the model could be more robust with more training time. For example, using a larger sample for hyperparameter search with more head and fine-tuning epochs per iteration could increase metrics. In addition, the final model tuning ran for 20 epochs, and accuracy increased until epoch 19. Tuning the model for more epochs could make the model a little more accurate.